# VoiceDub English - Zero-Spend GPU Backend

Run the English dubbing backend on Google Colab's **free T4 GPU** (15 GB VRAM).

**Zero-spend stack:**
- Local Whisper (faster-whisper on T4 GPU)
- Ollama or Google Translate (free)
- English dubbing rewrite (Ollama)
- Chatterbox-Turbo → XTTS v2 → Edge-TTS
- FFmpeg + Demucs assembly

**Setup:**
1. Go to **Runtime > Change runtime type > T4 GPU**
2. Run **Cell 1** (install) — this will restart the runtime
3. After restart, **skip Cell 1** and run Cells 2, 3, 4 in order

In [ ]:
#@title 1. Install Dependencies (run ONCE, restarts runtime)
#@markdown Run this cell first. After restart, skip this cell.

import os

# Clone or update repo
if os.path.exists('/content/app'):
    !cd /content/app && git fetch origin && git reset --hard origin/master
else:
    !git clone https://github.com/sasmalgiri/youtube-english-dubbing.git /content/app

# Install build tools
!apt-get -qq install -y libsndfile1 > /dev/null 2>&1

# Fix numpy for chatterbox
!pip install -q numpy==1.26.4

# Install chatterbox (Turbo + Multilingual)
!pip install -q "librosa>=0.11.0" "s3tokenizer" "torch>=2.6.0" "torchaudio>=2.6.0" \
    "transformers==4.46.3" "diffusers==0.29.0" "resemble-perth>=1.0.1" \
    "conformer>=0.3.2" "safetensors>=0.5.3" "spacy-pkuseg" "pykakasi>=2.3.0" \
    "pyloudnorm" "omegaconf"
!pip install -q chatterbox-tts --no-deps

# Install backend deps + pyngrok
!pip install -q fastapi uvicorn[standard] python-multipart pydantic edge-tts \
    faster-whisper deep-translator google-genai groq openai \
    sse-starlette rich yt-dlp pyngrok TTS

print("\n" + "=" * 60)
print("All installed! Restarting runtime for numpy fix...")
print("After restart, SKIP this cell and run Cells 2, 3, 4.")
print("=" * 60)
os._exit(0)

In [ ]:
#@title 2. Setup (Zero-Spend — no API keys required)
#@markdown All fields are OPTIONAL. The pipeline runs fully free without any keys.

NGROK_AUTH_TOKEN = "" #@param {type:"string"}
NGROK_DOMAIN = "" #@param {type:"string"}

#@markdown ---
#@markdown **Optional API keys** (speed boost, not required for zero-spend):
GROQ_API_KEY = "" #@param {type:"string"}
SAMBANOVA_API_KEY = "" #@param {type:"string"}
GEMINI_API_KEY = "" #@param {type:"string"}

#@markdown ---
#@markdown **Zero-spend pipeline (no keys needed):**
#@markdown - Whisper: local on T4 GPU
#@markdown - Translation: Google Translate (free)
#@markdown - TTS: Chatterbox-Turbo → XTTS v2 → Edge-TTS
#@markdown
#@markdown **With free-tier keys (faster):**
#@markdown - `GROQ_API_KEY`: Faster translation via Llama 3.3 70B
#@markdown - `SAMBANOVA_API_KEY`: Parallel translation engine
#@markdown - `NGROK_AUTH_TOKEN`: Required to expose backend to your local frontend

import os

# Set environment variables
if GROQ_API_KEY:
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY
if SAMBANOVA_API_KEY:
    os.environ['SAMBANOVA_API_KEY'] = SAMBANOVA_API_KEY
if GEMINI_API_KEY:
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# Ensure repo exists
if not os.path.exists('/content/app/backend'):
    !git clone https://github.com/sasmalgiri/youtube-english-dubbing.git /content/app

# Write .env file
os.makedirs('/content/app/backend', exist_ok=True)
with open('/content/app/backend/.env', 'w') as f:
    if GROQ_API_KEY:
        f.write(f'GROQ_API_KEY={GROQ_API_KEY}\n')
    if SAMBANOVA_API_KEY:
        f.write(f'SAMBANOVA_API_KEY={SAMBANOVA_API_KEY}\n')
    if GEMINI_API_KEY:
        f.write(f'GEMINI_API_KEY={GEMINI_API_KEY}\n')

# Install deno for yt-dlp
!curl -fsSL https://deno.land/install.sh | sh 2>/dev/null
os.environ['PATH'] = '/root/.deno/bin:' + os.environ.get('PATH', '')

# Check GPU
import torch
print(f"\nGPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({vram:.1f} GB VRAM)")
else:
    print("WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU")

# Pre-download Whisper model
print("\nPre-downloading Whisper model...")
%cd /content/app/backend
from faster_whisper import WhisperModel
device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute = 'float16' if device == 'cuda' else 'int8'
model = WhisperModel('large-v3-turbo', device=device, compute_type=compute)
del model

has_translation = GROQ_API_KEY or SAMBANOVA_API_KEY or GEMINI_API_KEY
translation = 'Groq' if GROQ_API_KEY else 'SambaNova' if SAMBANOVA_API_KEY else 'Gemini' if GEMINI_API_KEY else 'Google Translate (free)'

print("\n" + "=" * 60)
print("ZERO-SPEND ENGLISH DUBBING READY")
print(f"  GPU:         {gpu_name if torch.cuda.is_available() else 'CPU (slow)'}")
print(f"  Whisper:     Local on GPU (free)")
print(f"  Translation: {translation}")
print(f"  TTS:         Chatterbox-Turbo → XTTS v2 → Edge-TTS")
print(f"  ngrok:       {'configured' if NGROK_AUTH_TOKEN else 'MISSING (needed for frontend)'}")
print("=" * 60)
print("\nRun Cell 3 to start the server!")

In [ ]:
#@title 3. Start Backend Server + ngrok Tunnel
import subprocess
import time
import os
from pyngrok import ngrok, conf

os.environ['PATH'] = '/root/.deno/bin:' + os.environ.get('PATH', '')

if not NGROK_AUTH_TOKEN:
    raise ValueError('Missing NGROK_AUTH_TOKEN — go back to Cell 2')
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Start uvicorn
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/app/backend',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
time.sleep(3)

# Connect ngrok
if NGROK_DOMAIN:
    public_url = ngrok.connect(8000, 'http', domain=NGROK_DOMAIN)
    url = f'https://{NGROK_DOMAIN}'
else:
    public_url = ngrok.connect(8000, 'http')
    url = str(public_url)

print('=' * 60)
print('ENGLISH DUBBING BACKEND RUNNING ON FREE T4 GPU')
print(f'\nPUBLIC URL: {url}')
print(f'\nTest: {url}/api/health')
print('\nSet this in your frontend .env.local:')
print(f'  NEXT_PUBLIC_API_URL={url}')
print('=' * 60)

In [ ]:
#@title 4. Monitor Server Logs
#@markdown Keep this cell running to see backend logs.

import time

print('Monitoring server... Submit a job from your frontend.')
print('-' * 60)

try:
    while proc.poll() is None:
        line = proc.stdout.readline()
        if line:
            print(line.decode('utf-8', errors='replace').rstrip())
        else:
            time.sleep(0.5)
    print('\nServer process exited!')
except KeyboardInterrupt:
    print('\nStopped monitoring (server still running)')